# Moment Matrices and the Christoffel-Darboux Polynomial

This notebook explores the concept of moment matrices constructed from a small dataset of points. We will compare three different methods for constructing this matrix:

1.  **Empirical Measure**: The standard approach where the matrix is a sum of outer products of the feature vectors of the data points. This matrix is often rank-deficient.
2.  **Ball Measure (Smoothed)**: A "smoothed" version where we integrate the monomial features over a small ball around each data point. 
3.  **Gaussian Measure (Smoothed)**: Another smoothed version where we take the expectation of the monomial features with respect to a Gaussian distribution centered at each data point.

We will then compute and visualize the Christoffel-Darboux (CD) polynomial for each case. The CD polynomial, defined as $K(x,x) = \phi(x)^T M^{-1} \phi(x)$, provides a measure of leverage of point $x$. Its level sets can be interpreted as a data-driven geometry.

**Key Observation**: We expect the empirical matrix to be singular (rank-deficient), while the smoothed versions should be full-rank, leading to well-defined CD polynomials without explicit regularization.

## 1. Setup and Helper Functions

In [ ]:
import numpy as np
import itertools
from scipy.special import gamma as gamma_func, factorial2, comb
from scipy import integrate
import matplotlib.pyplot as plt
import seaborn as sns

N_DIM = 2      # n: Dimensionality of the space (1 or 2)
M_POINTS = 3   # m: Number of data points
DEGREE = 3     # d: Maximum degree of monomials

DATA_RADIUS = 1.0 # Radius of ball to sample points from
EPSILON = 0.5     # epsilon: Radius for the ball measure
LAMBDA_REG = 1e-6 # Regularization for the empirical matrix

In [ ]:
def get_monomial_basis(n_dim, degree):
    """Generates a list of multi-indices for monomials up to a given degree."""
    if n_dim == 1:
        return [(d,) for d in range(degree + 1)]
    
    basis = []
    for d_total in range(degree + 1):
        for exponents in itertools.product(range(d_total + 1), repeat=n_dim):
            if sum(exponents) == d_total:
                basis.append(exponents)
    # Sort for consistent ordering
    basis.sort(key=lambda x: (sum(x), x))
    return basis

def phi(x, basis):
    """Computes the monomial feature vector for a point x."""
    x = np.asarray(x)
    return np.array([np.prod(x**alpha) for alpha in basis])

def sample_points_from_ball(m, n, r):
    """Samples m points uniformly from an n-dimensional ball of radius r."""
    if n == 1:
        return (np.random.rand(m, 1) * 2 - 1) * r
    
    # Sample from a multivariate normal, then normalize and scale
    points = np.random.randn(m, n)
    norms = np.linalg.norm(points, axis=1, keepdims=True)
    unit_vectors = points / norms
    
    # Scale by a radius sampled to ensure uniformity
    radii = r * (np.random.rand(m, 1)**(1/n))
    return unit_vectors * radii

def get_grid(points, padding=1.0):
    """Creates a 2D grid for plotting."""
    x_min, y_min = points.min(axis=0) - padding
    x_max, y_max = points.max(axis=0) + padding
    grid_x, grid_y = np.meshgrid(np.linspace(x_min, x_max, 100),
                                 np.linspace(y_min, y_max, 100))
    return grid_x, grid_y

## 2. Data Generation

We generate $m=3$ points from a ball of radius $r=1$ in $n=2$ dimensions.

In [ ]:
np.random.seed(1)
points = sample_points_from_ball(M_POINTS, N_DIM, DATA_RADIUS)

plt.figure(figsize=(6, 6))
plt.scatter(points[:, 0], points[:, 1], s=100, label='Data Points', zorder=5)

# Draw circles for ball/gaussian measures
for p in points:
    ball_circle = plt.Circle(p, EPSILON, color='r', alpha=0.1, label='Ball Measure (ε)' if p is points[0] else "")
    gauss_circle = plt.Circle(p, EPSILON, color='b', alpha=0.1, label='Gaussian Measure (ε)' if p is points[0] else "")
    plt.gca().add_patch(ball_circle)
    plt.gca().add_patch(gauss_circle)

plt.title('Sampled Data Points and Measure Supports')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.axis('equal')
plt.legend()
plt.show()

## 3. Moment Matrix Implementations

We now create the monomial basis and then construct the three different moment matrices. We will check the rank of each matrix.

In [ ]:
# Get the monomial basis for our chosen dimension and degree
monomial_basis = get_monomial_basis(N_DIM, DEGREE)
basis_size = len(monomial_basis)
print(f"Dimension: {N_DIM}, Degree: {DEGREE}")
print(f"Monomial basis has {basis_size} elements:")
print(monomial_basis)

### 3.1. Empirical Moment Matrix

In [ ]:
M_emp = np.zeros((basis_size, basis_size))
for p in points:
    phi_p = phi(p, monomial_basis).reshape(-1, 1)
    M_emp += phi_p @ phi_p.T

rank_emp = np.linalg.matrix_rank(M_emp)
print(f"Empirical Moment Matrix Shape: {M_emp.shape}")
print(f"Rank of Empirical Matrix: {rank_emp} (Expected: <= {M_POINTS})")

We either use numerical integration (`scipy.integrate.nquad`) to compute them, or the analytic formula. 

### 3.2. Ball Measure Moment Matrix

Here, each entry $(\alpha, \beta)$ of the matrix is given by:
$$ [\tilde{M}_{d,\epsilon}]_{\alpha,\beta} = \sum_{i=1}^m \frac{1}{|B(x_i,\epsilon)|} \int_{B(x_i, \epsilon)} x^{\alpha+\beta} dx $$

In [ ]:
import numpy as np
import itertools
from scipy.special import gamma as gamma_func, factorial2, comb

# Helper to iterate through multi-indices k <= gamma
def iterate_sub_indices(gamma):
    ranges = [range(g + 1) for g in gamma]
    for k in itertools.product(*ranges):
        yield np.array(k)
        
def gaussian_moment(gamma, mu, sigma):
    """
    Calculates E[x^gamma] where x ~ N(mu, sigma^2 * I).
    This function directly implements the formula:
    E[x^gamma] = sum_{k<=gamma, k_j even} [ binom(gamma,k) * mu^(gamma-k) * E[y^k] ]
    where E[y^k] for y ~ N(0, sigma^2*I) is prod(sigma^k_j * (k_j-1)!!)
    """
    total_moment = 0
    mu = np.asarray(mu)
    gamma = np.asarray(gamma)
    
    for k in iterate_sub_indices(gamma):
        # The moment of a centered Gaussian is non-zero only if all exponents are even
        if all(val % 2 == 0 for val in k):
            # Binomial coefficient term: C(gamma_1, k_1) * C(gamma_2, k_2) * ...
            binom_prod = np.prod([comb(g, ki) for g, ki in zip(gamma, k)])
            
            # Term for the mean: mu_1^(gamma_1-k_1) * mu_2^(gamma_2-k_2) * ...
            mu_term = np.prod(mu**(gamma - k))
            
            # Term for the centered moment E[y^k]
            sigma_term = 1.0
            for ki in k:
                if ki > 0:
                    sigma_term *= (sigma**ki) * factorial2(ki - 1) # (k-1)!! is the double factorial

            total_moment += binom_prod * mu_term * sigma_term
            
    return total_moment

def integral_monomial_centered_ball(k, epsilon, n_dim):
    """
    Calculates Integral_{B(0,eps)} y^k dy.
    The integral is non-zero only if all components of k are even.
    If k=2m, the formula is:
    eps^(n+|2m|) * [ prod_{j=1..n} Gamma(m_j + 1/2) ] / Gamma(n/2 + |m| + 1)
    """
    k = np.asarray(k)
    # Integral is 0 if any exponent is odd due to symmetry
    if not all(val % 2 == 0 for val in k):
        return 0.0
    
    m = k / 2
    abs_m = np.sum(m)
    abs_2m = np.sum(k)

    numerator = np.prod([gamma_func(mj + 0.5) for mj in m])
    denominator = gamma_func(n_dim/2 + abs_m + 1)
    
    return (epsilon**(n_dim + abs_2m)) * (numerator / denominator)

def ball_moment(gamma, center, epsilon, n_dim):
    """
    Calculates Integral_{B(center,eps)} x^gamma dx using multinomial expansion.
    The formula is:
    Integral = sum_{k<=gamma} [ binom(gamma,k) * center^(gamma-k) * Integral_{B(0,eps)} y^k dy ]
    """
    total_integral = 0
    center = np.asarray(center)
    gamma = np.asarray(gamma)

    for k in iterate_sub_indices(gamma):
        binom_prod = np.prod([comb(g, ki) for g, ki in zip(gamma, k)])
        center_term = np.prod(center**(gamma - k))
        integral_term = integral_monomial_centered_ball(k, epsilon, n_dim)
        
        total_integral += binom_prod * center_term * integral_term
        
    return total_integral

M_gauss_analytic = np.zeros((basis_size, basis_size))
M_ball_analytic = np.zeros((basis_size, basis_size))
ball_volume = (np.pi**(N_DIM/2) / gamma_func(N_DIM/2 + 1)) * (EPSILON**N_DIM)

print("Building matrices with analytical formulas...")
for i, alpha in enumerate(monomial_basis):
    for j, beta in enumerate(monomial_basis):
        if i > j: continue # Matrix is symmetric
            
        gamma = tuple(a + b for a, b in zip(alpha, beta))
        
        # --- Gaussian Case ---
        total_gauss_moment = sum(gaussian_moment(gamma, p, EPSILON) for p in points)
        M_gauss_analytic[i, j] = total_gauss_moment
        
        # --- Ball Case ---
        total_ball_integral = sum(ball_moment(gamma, p, EPSILON, N_DIM) for p in points)
        M_ball_analytic[i, j] = total_ball_integral / ball_volume

# Fill in the symmetric lower-triangle part of the matrices
if basis_size > 0:
    i_lower = np.tril_indices(basis_size, -1)
    M_gauss_analytic[i_lower] = M_gauss_analytic.T[i_lower]
    M_ball_analytic[i_lower] = M_ball_analytic.T[i_lower]
    
print("Done.")

print(M_gauss_analytic.shape, np.linalg.matrix_rank(M_gauss_analytic))
print(M_ball_analytic.shape, np.linalg.matrix_rank(M_ball_analytic))

# You can now use M_gauss_analytic and M_ball_analytic for the rest of the analysis.
# They will be much faster to compute and more accurate.

In [ ]:
# This function defines the integrand for a given monomial
def monomial_integrand(gamma):
    def integrand(*args):
        val = 1.0
        for i in range(len(args)):
            val *= args[i] ** gamma[i]
        return val
    return integrand

# (x-xi)**2 + (y-yi)**2 = eps**2
# y = y_i +- sqrt(eps**2 - (x-xi)**2)
def limits_y(p):
    return [p[1] - EPSILON, p[1] + EPSILON] 

def limits_x_given_y(p):
    return lambda y: [
        float(p[0] - np.sqrt(max(1e-10, EPSILON**2 - (y - p[1])**2))), 
        float(p[0] + np.sqrt(max(1e-10, EPSILON**2 - (y - p[1])**2)))
    ]
    
p = [0, 0]
# note that the order here is crucial. see nquad documentation. 
ranges = [
    limits_x_given_y(p), # calcualte x as a function of y.
    limits_y(p)  # choose y in the right window
]

# sanity check: when using gamma=(0,0), we should just get the volume of the sphere
integral, _ = integrate.nquad(monomial_integrand((0, 0)), ranges)
print(integral)
print(np.pi * EPSILON**2)

In [ ]:
M_ball = np.zeros((basis_size, basis_size))
ball_volume = (np.pi**(N_DIM/2) / gamma_func(N_DIM/2 + 1)) * (EPSILON**N_DIM)

print("Building Ball Measure Matrix (this may take a moment)...")
for i, alpha in enumerate(monomial_basis):
    for j, beta in enumerate(monomial_basis):
        if i > j: # Matrix is symmetric
            M_ball[i, j] = M_ball[j, i]
            continue
        
        gamma = tuple(a + b for a, b in zip(alpha, beta))
        
        total_integral = 0
        for p in points:
            if N_DIM == 1:
                 ranges = [[p[0] - EPSILON, p[0] + EPSILON]]
            else: # N_DIM == 2 or 3                    
                ranges = [
                    limits_x_given_y(p), # calcualte y as a function of x.
                    limits_y(p)
                ]
            # function to integrate: x^gamma
            integral, _ = integrate.nquad(monomial_integrand(gamma), ranges)
            total_integral += integral
            
        M_ball[i, j] = total_integral / ball_volume

rank_ball = np.linalg.matrix_rank(M_ball)
print("Done.")
print(f"Ball Measure Moment Matrix Shape: {M_ball.shape}")
print(f"Rank of Ball Measure Matrix: {rank_ball} (Expected: {basis_size})")

### 3.3. Gaussian Measure Moment Matrix

Here, each entry $(\alpha, \beta)$ is the sum of expectations:
$$ [\tilde{M}_{d,\sigma}]_{\alpha,\beta} = \sum_{i=1}^m E_{x \sim N(x_i, \sigma^2 I)}[x^{\alpha+\beta}] $$

In [ ]:
# Helper to iterate through multi-indices k <= gamma
def iterate_sub_indices(gamma):
    ranges = [range(g + 1) for g in gamma]
    for k in itertools.product(*ranges):
        yield k

def gaussian_moment(gamma, mu, sigma):
    """Calculates E[x^gamma] where x ~ N(mu, sigma^2 * I)."""
    total_moment = 0
    mu = np.asarray(mu)
    for k in iterate_sub_indices(gamma):
        # Check if all components of k are even
        if all(val % 2 == 0 for val in k):
            binom_prod = np.prod([comb(g, ki) for g, ki in zip(gamma, k)])
            mu_term = np.prod(mu**(np.array(gamma) - np.array(k)))
            
            sigma_term = 1.0
            for ki in k:
                if ki > 0:
                    sigma_term *= (sigma**ki) * factorial2(ki - 1)

            total_moment += binom_prod * mu_term * sigma_term
    return total_moment

M_gauss = np.zeros((basis_size, basis_size))

print("Building Gaussian Measure Matrix...")
for i, alpha in enumerate(monomial_basis):
    for j, beta in enumerate(monomial_basis):
        if i > j:
            M_gauss[i, j] = M_gauss[j, i]
            continue

        gamma = tuple(a + b for a, b in zip(alpha, beta))
        total_moment = sum(gaussian_moment(gamma, p, EPSILON) for p in points)
        M_gauss[i, j] = total_moment

rank_gauss = np.linalg.matrix_rank(M_gauss)
print("Done.")
print(f"Gaussian Measure Moment Matrix Shape: {M_gauss.shape}")
print(f"Rank of Gaussian Measure Matrix: {rank_gauss} (Expected: {basis_size})")

## 4. Christoffel-Darboux Polynomial Calculation and Visualization

The CD polynomial is $K(x,x) = \phi(x)^T M^{-1} \phi(x)$. We will calculate the inverse of each matrix (regularizing the empirical one) and then evaluate this expression over a grid to visualize the result.

In [ ]:
def calculate_cd_polynomial(M, basis, grid_x, grid_y, regularize=False, lambda_reg=1e-6):
    """Calculates the CD polynomial on a grid."""
    if regularize:
        M_inv = np.linalg.inv(M + lambda_reg * np.eye(M.shape[0]))
    else:
        M_inv = np.linalg.inv(M)
        
    Z = np.zeros_like(grid_x)
    for i in range(grid_x.shape[0]):
        for j in range(grid_x.shape[1]):
            x_point = [grid_x[i, j], grid_y[i, j]]
            phi_x = phi(x_point, basis)
            Z[i, j] = phi_x.T @ M_inv @ phi_x
    return Z

def plot_cd_polynomial(Z, grid_x, grid_y, points, title):
    """Plots the CD polynomial as a contour plot."""
    plt.contourf(grid_x, grid_y, np.log10(Z), levels=50, cmap='viridis')
    plt.colorbar(label='$K(x,x)$')
    plt.scatter(points[:, 0], points[:, 1], c='red', s=10, zorder=5, label='Data Points')
    plt.title(title)
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    plt.axis('equal')
    plt.axis("off")
    plt.legend()

In [ ]:
grid_x, grid_y = get_grid(points, padding=1.5)

# Calculate CD polynomials for each case
print("Calculating CD Polynomials...")
Z_emp = calculate_cd_polynomial(M_emp, monomial_basis, grid_x, grid_y, regularize=True, lambda_reg=LAMBDA_REG)
Z_ball = calculate_cd_polynomial(M_ball, monomial_basis, grid_x, grid_y)
Z_gauss = calculate_cd_polynomial(M_gauss, monomial_basis, grid_x, grid_y)
Z_ball_an = calculate_cd_polynomial(M_ball_analytic, monomial_basis, grid_x, grid_y)
Z_gauss_an = calculate_cd_polynomial(M_gauss_analytic, monomial_basis, grid_x, grid_y)
print("Done.")

# Plotting
fig, axes = plt.subplots(1, 5, figsize=(21, 4))

plt.sca(axes[0])
plot_cd_polynomial(Z_emp, grid_x, grid_y, points, f'Empirical')

plt.sca(axes[1])
plot_cd_polynomial(Z_ball, grid_x, grid_y, points, f'Ball numerical')

plt.sca(axes[2])
plot_cd_polynomial(Z_gauss, grid_x, grid_y, points, f'Gaussian numerical')

plt.sca(axes[3])
plot_cd_polynomial(Z_ball_an, grid_x, grid_y, points, f'Ball analytic')

plt.sca(axes[4])
plot_cd_polynomial(Z_gauss_an, grid_x, grid_y, points, f'Gaussian analytic')

plt.tight_layout()
plt.show()

## 5. Conclusion

As expected, the results demonstrate the following:

- **Rank**: The empirical moment matrix is rank-deficient (rank 3, equal to the number of points), as it is constructed from only 3 independent vectors in a 10-dimensional feature space. In contrast, both the ball and Gaussian measure matrices are full-rank (rank 10). This is because the integration/expectation step introduces new information and breaks the linear dependencies.

- **Christoffel-Darboux Polynomial**: 
    - The CD polynomial for the **empirical case** is heavily influenced by the regularization parameter and its shape is highly concentrated around the data points, quickly decaying to zero elsewhere. The geometry it defines is somewhat degenerate.
    - The CD polynomials for the **smoothed measures** are well-behaved and provide a much more stable and smooth geometry. The shape of their level sets envelops all the data points, defining a region of interest based on the data. The `epsilon` parameter controls the "tightness" of this shape around the points.